# Lab 8 — Análise Exploratória de Dados

## Objetivo

Este laboratório utiliza a camada Silver para responder a cinco perguntas de negócio relacionadas ao perfil dos clientes, risco de fraude, comportamento temporal e oportunidades comerciais.

Para cada análise, será utilizado o framework:

**Finding → Insight → Ação**

- **Finding:** resultado numérico encontrado nos dados;
- **Insight:** interpretação do resultado para o negócio;
- **Ação:** decisão ou iniciativa sugerida a partir do achado.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "silver").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "silver").exists():
            pasta_projeto = pasta_pai
            break

arquivo_silver = (
    pasta_projeto
    / "dados"
    / "silver"
    / "transactions_enriched.parquet"
)

arquivo_clientes_bronze = (
    pasta_projeto
    / "dados"
    / "bronze"
    / "customers.parquet"
)

arquivo_banco = (
    pasta_projeto
    / "dia2_transformacao"
    / "lab08_eda"
    / "eda.duckdb"
)

assert arquivo_silver.exists(), "Arquivo Silver não encontrado."
assert arquivo_clientes_bronze.exists(), (
    "Bronze de clientes não encontrada."
)

print("Silver:", arquivo_silver)
print("Clientes Bronze:", arquivo_clientes_bronze)

Silver: C:\BigData\bigdata-curso-gabriel\dados\silver\transactions_enriched.parquet
Clientes Bronze: C:\BigData\bigdata-curso-gabriel\dados\bronze\customers.parquet


In [2]:
conexao = duckdb.connect(str(arquivo_banco))

caminho_silver = (
    arquivo_silver.as_posix().replace("'", "''")
)

caminho_clientes = (
    arquivo_clientes_bronze.as_posix().replace("'", "''")
)

conexao.execute(f"""
    CREATE OR REPLACE TABLE silver_transactions AS
    SELECT *
    FROM read_parquet('{caminho_silver}')
""")

conexao.execute(f"""
    CREATE OR REPLACE TABLE bronze_customers AS
    SELECT *
    FROM read_parquet('{caminho_clientes}')
""")

validacao_inicial = conexao.execute("""
    SELECT
        COUNT(*) AS transacoes,
        COUNT(DISTINCT customer_id) AS clientes_ativos,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes,
        ROUND(
            100.0 * AVG(
                CASE WHEN is_fraud THEN 1 ELSE 0 END
            ),
            2
        ) AS taxa_fraude_pct
    FROM silver_transactions
""").df()

validacao_inicial

,transacoes,clientes_ativos,fraudes,taxa_fraude_pct
0,100000,9104,1833.0,1.83


## 1. Segmentação por credit score

A primeira análise compara o credit score médio e a quantidade de clientes ativos em cada segmento. O objetivo é verificar se a classificação dos segmentos é coerente com o perfil de crédito observado.

In [3]:
analise_segmento = conexao.execute("""
    SELECT
        segment,
        ROUND(AVG(credit_score), 1) AS score_medio,
        MIN(credit_score) AS menor_score,
        MAX(credit_score) AS maior_score,
        COUNT(DISTINCT customer_id) AS clientes
    FROM silver_transactions
    GROUP BY segment
    ORDER BY score_medio
""").df()

analise_segmento

,segment,score_medio,menor_score,maior_score,clientes
0,Standard,643.9,300,900,2665
1,Premium,652.7,300,900,5563
2,High-Risk,659.8,300,900,876


## 2. Risco por tipo de transação

Esta análise compara a quantidade e a taxa de fraude entre compras, saques, transferências e pagamentos. A taxa é mais apropriada do que a quantidade isolada, pois considera o volume diferente de cada tipo de transação.

In [4]:
analise_tipo_transacao = conexao.execute("""
    SELECT
        transaction_type,
        COUNT(*) AS total_transacoes,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            AS fraudes,
        ROUND(
            100.0
            * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS taxa_fraude_pct,
        ROUND(AVG(amount), 2) AS ticket_medio,
        ROUND(
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END),
            2
        ) AS valor_em_risco
    FROM silver_transactions
    GROUP BY transaction_type
    ORDER BY taxa_fraude_pct DESC
""").df()

analise_tipo_transacao

,transaction_type,total_transacoes,fraudes,taxa_fraude_pct,ticket_medio,valor_em_risco
0,transferencia,20102,390.0,1.94,184.37,70295.25
1,pagamento,10180,196.0,1.93,184.79,31433.05
2,compra,49782,929.0,1.87,183.35,154551.29
3,saque,19936,318.0,1.60,183.81,58308.33


## 3. Padrão temporal por dia do mês

A análise verifica quais dias do mês apresentam mais fraudes e se esse resultado está associado a um maior volume de transações. A comparação entre quantidade e taxa evita interpretar um dia como mais arriscado apenas porque possui mais movimentações.

In [5]:
analise_dia_mes = conexao.execute("""
    SELECT
        day AS dia_do_mes,
        COUNT(*) AS total_transacoes,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            AS fraudes,
        ROUND(
            100.0
            * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS taxa_fraude_pct
    FROM silver_transactions
    GROUP BY day
    ORDER BY fraudes DESC, taxa_fraude_pct DESC
""").df()

analise_dia_mes.head(10)

,dia_do_mes,total_transacoes,fraudes,taxa_fraude_pct
0,21,7777,161.0,2.07
1,26,7913,157.0,1.98
2,24,7792,148.0,1.90
3,27,7801,144.0,1.85
4,25,7689,140.0,1.82
5,20,7850,140.0,1.78
6,23,7696,133.0,1.73
7,22,7838,132.0,1.68
8,28,7914,131.0,1.66
9,11,1505,41.0,2.72


In [6]:
maiores_por_volume = analise_dia_mes.sort_values(
    ["total_transacoes", "fraudes"],
    ascending=False
).head(10)

maiores_por_taxa = analise_dia_mes.sort_values(
    ["taxa_fraude_pct", "fraudes"],
    ascending=False
).head(10)

print("Dez dias com maior volume:")
display(maiores_por_volume)

print("Dez dias com maior taxa de fraude:")
display(maiores_por_taxa)

Dez dias com maior volume:


,dia_do_mes,total_transacoes,fraudes,taxa_fraude_pct
8,28,7914,131.0,1.66
1,26,7913,157.0,1.98
5,20,7850,140.0,1.78
7,22,7838,132.0,1.68
3,27,7801,144.0,1.85
2,24,7792,148.0,1.90
0,21,7777,161.0,2.07
6,23,7696,133.0,1.73
4,25,7689,140.0,1.82
19,8,1655,29.0,1.75


Dez dias com maior taxa de fraude:


,dia_do_mes,total_transacoes,fraudes,taxa_fraude_pct
9,11,1505,41.0,2.72
10,14,1553,35.0,2.25
11,3,1551,34.0,2.19
0,21,7777,161.0,2.07
12,17,1605,33.0,2.06
13,7,1556,32.0,2.06
15,9,1544,31.0,2.01
1,26,7913,157.0,1.98
14,16,1632,32.0,1.96
16,15,1584,31.0,1.96


## 4. Cohort de clientes antigos e novos

Os clientes serão classificados de acordo com o tempo decorrido entre sua data de cadastro e a data de execução da análise.

Será considerado antigo o cliente cadastrado há mais de 365 dias. Como esse resultado depende da data atual, a data de referência será apresentada junto com os indicadores.

In [7]:
data_referencia = conexao.execute("""
    SELECT CURRENT_DATE
""").fetchone()[0]

analise_cohort = conexao.execute("""
    SELECT
        CASE
            WHEN DATEDIFF(
                'day',
                c.created_at,
                CURRENT_DATE
            ) > 365
                THEN 'cliente antigo'
            ELSE 'cliente novo'
        END AS cohort,

        COUNT(*) AS transacoes,
        COUNT(DISTINCT t.customer_id) AS clientes,
        ROUND(AVG(t.amount), 2) AS ticket_medio,

        SUM(CASE WHEN t.is_fraud THEN 1 ELSE 0 END)
            AS fraudes,

        ROUND(
            100.0
            * SUM(CASE WHEN t.is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS taxa_fraude_pct

    FROM silver_transactions t
    INNER JOIN bronze_customers c
        ON t.customer_id = c.customer_id

    GROUP BY cohort
    ORDER BY cohort
""").df()

print("Data de referência:", data_referencia)
analise_cohort

Data de referência: 2026-09-15


,cohort,transacoes,clientes,ticket_medio,fraudes,taxa_fraude_pct
0,cliente antigo,56019,5041,182.73,1000.0,1.79
1,cliente novo,43981,4063,185.14,833.0,1.89


## 5. Clientes Premium com alta frequência de compras

A última análise identifica clientes Premium com mais de dez transações do tipo compra. Esse público pode representar uma oportunidade para campanhas direcionadas, programas de benefícios ou ofertas relacionadas ao comportamento de consumo.

In [8]:
resumo_cross_sell = conexao.execute("""
    WITH clientes_elegiveis AS (
        SELECT
            customer_id,
            COUNT(*) AS compras,
            ROUND(SUM(amount), 2) AS valor_compras,
            ROUND(AVG(amount), 2) AS ticket_medio
        FROM silver_transactions
        WHERE segment = 'Premium'
          AND transaction_type = 'compra'
        GROUP BY customer_id
        HAVING COUNT(*) > 10
    )

    SELECT
        COUNT(*) AS clientes_elegiveis,
        MIN(compras) AS menor_quantidade_compras,
        MAX(compras) AS maior_quantidade_compras,
        ROUND(AVG(compras), 2) AS media_compras
    FROM clientes_elegiveis
""").df()

resumo_cross_sell

,clientes_elegiveis,menor_quantidade_compras,maior_quantidade_compras,media_compras
0,805,11,49,16.23


In [9]:
clientes_cross_sell = conexao.execute("""
    SELECT
        customer_id,
        COUNT(*) AS compras,
        ROUND(SUM(amount), 2) AS valor_compras,
        ROUND(AVG(amount), 2) AS ticket_medio
    FROM silver_transactions
    WHERE segment = 'Premium'
      AND transaction_type = 'compra'
    GROUP BY customer_id
    HAVING COUNT(*) > 10
    ORDER BY compras DESC, valor_compras DESC
    LIMIT 10
""").df()

clientes_cross_sell

,customer_id,compras,valor_compras,ticket_medio
0,1920,49,9144.18,186.62
1,8500,44,8600.39,195.46
2,7531,41,7597.47,185.30
3,9306,40,8883.46,222.09
4,5251,40,6256.64,156.42
5,7105,39,10999.67,282.04
6,5354,39,8473.80,217.28
7,1072,39,6605.25,169.37
8,3686,38,12016.22,316.22
9,8633,37,5401.80,145.99


In [10]:
pd.set_option("display.max_colwidth", None)

sintese_analises = pd.DataFrame({
    "Análise": [
        "Segmentação por score",
        "Risco por tipo de transação",
        "Padrão por dia do mês",
        "Clientes antigos versus novos",
        "Cross-sell Premium"
    ],

    "Finding": [
        (
            "O segmento High-Risk apresentou score médio de 659,8, "
            "superior ao Premium, com 652,7, e ao Standard, com "
            "643,9. Todos os segmentos possuem scores entre 300 e 900."
        ),
        (
            "Transferências apresentaram a maior taxa de fraude, "
            "com 1,94%. Compras concentraram 929 fraudes e "
            "R$ 154.551,29 em valor de risco."
        ),
        (
            "O dia 21 apresentou 161 fraudes em 7.777 transações, "
            "com taxa de 2,07%. Os dias 20 a 28 concentraram os "
            "maiores números absolutos de fraude."
        ),
        (
            "Clientes novos apresentaram ticket médio de R$ 185,14 "
            "e taxa de fraude de 1,89%. Entre os antigos, os resultados "
            "foram R$ 182,73 e 1,79%."
        ),
        (
            "Foram identificados 805 clientes Premium com mais de "
            "dez compras. A média foi de 16,23 compras e o máximo "
            "foi de 49."
        )
    ],

    "Insight": [
        (
            "O credit score isoladamente não explica a classificação "
            "dos segmentos da base."
        ),
        (
            "Transferências possuem o maior risco relativo, enquanto "
            "compras representam a maior exposição absoluta devido "
            "ao seu volume."
        ),
        (
            "A concentração no fim do mês parece decorrer principalmente "
            "do maior volume transacional, e não apenas de uma taxa "
            "de fraude superior."
        ),
        (
            "As diferenças são pequenas e não indicam que o tempo de "
            "relacionamento, isoladamente, determine o valor transacionado "
            "ou o risco."
        ),
        (
            "Existe um público Premium com alta frequência de consumo, "
            "mas o grupo é amplo para uma abordagem individual."
        )
    ],

    "Ação proposta": [
        (
            "Investigar as variáveis que determinam o segmento e combinar "
            "credit score, comportamento transacional e histórico de risco."
        ),
        (
            "Aplicar controles adicionais em transferências e pagamentos "
            "de maior risco, mantendo o monitoramento de compras pela "
            "exposição absoluta."
        ),
        (
            "Reforçar o monitoramento entre os dias 20 e 28 e criar alertas "
            "que considerem taxa de fraude e volume mínimo."
        ),
        (
            "Não diferenciar políticas apenas pela antiguidade. Combinar "
            "essa informação com segmento, risk score e comportamento."
        ),
        (
            "Automatizar uma campanha segmentada e priorizar os clientes "
            "com maior frequência e maior valor total de compras."
        )
    ]
})

sintese_analises

,Análise,Finding,Insight,Ação proposta
0,Segmentação por score,"O segmento High-Risk apresentou score médio de 659,8, superior ao Premium, com 652,7, e ao Standard, com 643,9. Todos os segmentos possuem scores entre 300 e 900.",O credit score isoladamente não explica a classificação dos segmentos da base.,"Investigar as variáveis que determinam o segmento e combinar credit score, comportamento transacional e histórico de risco."
1,Risco por tipo de transação,"Transferências apresentaram a maior taxa de fraude, com 1,94%. Compras concentraram 929 fraudes e R$ 154.551,29 em valor de risco.","Transferências possuem o maior risco relativo, enquanto compras representam a maior exposição absoluta devido ao seu volume.","Aplicar controles adicionais em transferências e pagamentos de maior risco, mantendo o monitoramento de compras pela exposição absoluta."
2,Padrão por dia do mês,"O dia 21 apresentou 161 fraudes em 7.777 transações, com taxa de 2,07%. Os dias 20 a 28 concentraram os maiores números absolutos de fraude.","A concentração no fim do mês parece decorrer principalmente do maior volume transacional, e não apenas de uma taxa de fraude superior.",Reforçar o monitoramento entre os dias 20 e 28 e criar alertas que considerem taxa de fraude e volume mínimo.
3,Clientes antigos versus novos,"Clientes novos apresentaram ticket médio de R$ 185,14 e taxa de fraude de 1,89%. Entre os antigos, os resultados foram R$ 182,73 e 1,79%.","As diferenças são pequenas e não indicam que o tempo de relacionamento, isoladamente, determine o valor transacionado ou o risco.","Não diferenciar políticas apenas pela antiguidade. Combinar essa informação com segmento, risk score e comportamento."
4,Cross-sell Premium,"Foram identificados 805 clientes Premium com mais de dez compras. A média foi de 16,23 compras e o máximo foi de 49.","Existe um público Premium com alta frequência de consumo, mas o grupo é amplo para uma abordagem individual.",Automatizar uma campanha segmentada e priorizar os clientes com maior frequência e maior valor total de compras.


In [11]:
conexao.close()

print("Conexão encerrada.")
print("Consultas do Lab 8 executadas com sucesso.")

Conexão encerrada.
Consultas do Lab 8 executadas com sucesso.


## Conclusão

A análise exploratória mostrou que o risco deve ser avaliado por diferentes perspectivas. O segmento High-Risk apresentou a maior taxa de fraude, embora seu credit score médio não tenha sido inferior ao dos demais segmentos. Isso demonstra que o score de crédito não explica isoladamente a classificação de risco.

Por tipo de transação, transferências apresentaram a maior taxa relativa, enquanto compras concentraram a maior quantidade de fraudes e o maior valor financeiro em risco. Temporalmente, os dias 20 a 28 apresentaram mais ocorrências, acompanhando o aumento do volume de transações.

A comparação entre clientes antigos e novos revelou diferenças pequenas, insuficientes para justificar políticas baseadas apenas no tempo de relacionamento. Por outro lado, foram identificados 805 clientes Premium com elevada frequência de compras, formando um público relevante para campanhas automatizadas e ações comerciais direcionadas.